# 🐄 Détection de Chaleur Bovine — Moment Optimal d'Insémination
**Import direct des données → Features → XGBoost → Évaluation**

Données utilisées : `cbt`, `milk`, `ankle`, `behavior_labels`

## 0. Installation des librairies

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn xgboost imbalanced-learn --quiet


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import glob, os, warnings
from datetime import timezone, timedelta
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings('ignore')
print('✅ Imports OK')


## 2. Chemins des données

> ⚙️ **Modifier `BASE` si nécessaire**

In [ ]:
BASE_MAIN     = r'C:\Users\USER\Downloads\sensor_data\sensor_data\main_data'
BASE_BEHAVIOR = r'C:\Users\USER\Downloads\sensor_data\sensor_data\behavior_labels\individual'

CDT = timezone(timedelta(hours=-5))

def to_dt(ts):
    return pd.to_datetime(ts, unit='s', utc=True).dt.tz_convert(CDT)

print('📂 Chemins définis')
print(f'   main_data      : {BASE_MAIN}')
print(f'   behavior_labels: {BASE_BEHAVIOR}')


## 3. Chargement des données

### 3.1 Behavior labels

In [ ]:
frames = []
for f in sorted(glob.glob(f'{BASE_BEHAVIOR}/C*.csv')):
    cow_id = os.path.basename(f).split('_')[0]
    df_tmp = pd.read_csv(f)
    df_tmp['cow_id']   = cow_id
    df_tmp['datetime'] = to_dt(df_tmp['timestamp'])
    df_tmp['date']     = df_tmp['datetime'].dt.date
    df_tmp['hour']     = df_tmp['datetime'].dt.hour
    frames.append(df_tmp)
beh = pd.concat(frames, ignore_index=True)
print(f'✅ Behavior : {beh.shape}')
beh.head(3)


### 3.2 CBT (température corporelle)

In [ ]:
frames = []
for f in sorted(glob.glob(f'{BASE_MAIN}/cbt/C*.csv')):
    cow_id = os.path.basename(f).replace('.csv', '')
    df_tmp = pd.read_csv(f)
    df_tmp['cow_id']   = cow_id
    df_tmp['datetime'] = to_dt(df_tmp['timestamp'])
    df_tmp['date']     = df_tmp['datetime'].dt.date
    df_tmp['hour']     = df_tmp['datetime'].dt.hour
    frames.append(df_tmp)
cbt = pd.concat(frames, ignore_index=True)
print(f'✅ CBT : {cbt.shape}')
cbt.head(3)


### 3.3 Ankle (capteur cheville)

In [ ]:
frames = []
for f in sorted(glob.glob(f'{BASE_MAIN}/ankle/**/*.csv', recursive=True)):
    cow_id = os.path.basename(f).split('_')[0]
    df_tmp = pd.read_csv(f)
    df_tmp['cow_id']   = cow_id
    df_tmp['datetime'] = to_dt(df_tmp['timestamp'])
    df_tmp['date']     = df_tmp['datetime'].dt.date
    df_tmp['hour']     = df_tmp['datetime'].dt.hour
    frames.append(df_tmp)
ankle = pd.concat(frames, ignore_index=True)
print(f'✅ Ankle : {ankle.shape}')
ankle.head(3)


### 3.4 Milk (production laitière)

In [ ]:
frames = []
for f in sorted(glob.glob(f'{BASE_MAIN}/milk/C*.csv')):
    cow_id = os.path.basename(f).replace('.csv', '')
    df_tmp = pd.read_csv(f)
    df_tmp['cow_id']   = cow_id
    df_tmp['datetime'] = to_dt(df_tmp['timestamp'])
    df_tmp['date']     = df_tmp['datetime'].dt.date
    frames.append(df_tmp)
milk = pd.concat(frames, ignore_index=True)
print(f'✅ Milk : {milk.shape}')
milk.head(3)


## 4. Feature Engineering

### 4.1 Agrégation comportement horaire + label

In [ ]:
beh_h = beh.groupby(['cow_id','date','hour']).agg(
    mounting_count = ('behavior', lambda x: (x==6).sum()),
    walking_count  = ('behavior', lambda x: (x==1).sum()),
    standing_count = ('behavior', lambda x: (x==3).sum()),
    lying_count    = ('behavior', lambda x: (x==2).sum()),
    feeding_count  = ('behavior', lambda x: (x==4).sum()),
    total_obs      = ('behavior', 'count')
).reset_index()

beh_h['walking_h']  = beh_h['walking_count']  / 3600
beh_h['standing_h'] = beh_h['standing_count'] / 3600
beh_h['lying_h']    = beh_h['lying_count']    / 3600
beh_h['feeding_h']  = beh_h['feeding_count']  / 3600
beh_h['activity']   = beh_h['walking_count']  + beh_h['standing_count']

# LABEL : 1 = chaleur (mounting observé)
beh_h['label'] = (beh_h['mounting_count'] > 0).astype(int)

print('📌 Distribution labels:')
print(beh_h['label'].value_counts())


### 4.2 CBT horaire

In [ ]:
f_cbt = cbt.groupby(['cow_id','date','hour'])['temperature_C'].agg(
    cbt_mean='mean', cbt_max='max', cbt_std='std', cbt_min='min'
).reset_index()
print(f'✅ CBT horaire : {f_cbt.shape}')


### 4.3 Ankle horaire

In [ ]:
f_ankle = ankle.groupby(['cow_id','date','hour'])['lying'].agg(
    lying_h_ankle    = lambda x: x.sum() / 60,
    standing_h_ankle = lambda x: (1 - x).sum() / 60,
    activity_ankle   = 'count'
).reset_index()
print(f'✅ Ankle horaire : {f_ankle.shape}')


### 4.4 Milk journalier

In [ ]:
f_milk = milk.groupby(['cow_id','date'])['milk_weight_kg'].mean().reset_index()
f_milk.rename(columns={'milk_weight_kg': 'milk_kg'}, inplace=True)
print(f'✅ Milk journalier : {f_milk.shape}')


### 4.5 Merge final

In [ ]:
df = beh_h[['cow_id','date','hour',
            'walking_h','standing_h','lying_h','feeding_h',
            'activity','mounting_count','label']].copy()

df = df.merge(f_cbt,   on=['cow_id','date','hour'], how='inner')
df = df.merge(f_ankle, on=['cow_id','date','hour'], how='left')
df = df.merge(f_milk,  on=['cow_id','date'],        how='left')

# Filtrer CBT invalides
df = df[df['cbt_mean'] > 35.0].reset_index(drop=True)
df = df.fillna(0)

print(f'✅ Dataset après merge : {df.shape}')
print(f'   cbt_mean min/max : {df["cbt_mean"].min():.2f} / {df["cbt_mean"].max():.2f}°C')
print(f'\n📌 Labels après merge:')
print(df['label'].value_counts())
df.head()


### 4.6 Seuils physiologiques + score estrus

In [ ]:
df['cbt_above_threshold'] = (df['cbt_mean'] > 38.8).astype(int)
df['cbt_high']            = (df['cbt_mean'] > 39.0).astype(int)
df['cbt_fever']           = (df['cbt_mean'] > 39.5).astype(int)

act_q75 = df['activity'].quantile(0.75)
act_q90 = df['activity'].quantile(0.90)
print(f'activity Q75={act_q75:.0f}  Q90={act_q90:.0f}')

df['high_activity_75'] = (df['activity'] > act_q75).astype(int)
df['high_activity_90'] = (df['activity'] > act_q90).astype(int)
df['high_walking']     = (df['walking_h']  > 0.20).astype(int)
df['high_standing']    = (df['standing_h'] > 0.50).astype(int)
df['low_lying']        = (df['lying_h']    < 0.30).astype(int)
df['low_feeding']      = (df['feeding_h']  < 0.10).astype(int)

cow_milk_mean      = df.groupby('cow_id')['milk_kg'].transform('mean')
df['milk_drop_10'] = (df['milk_kg'] < cow_milk_mean * 0.90).astype(int)
df['milk_drop_15'] = (df['milk_kg'] < cow_milk_mean * 0.85).astype(int)
df['is_night']     = df['hour'].apply(lambda h: 1 if h >= 22 or h <= 6 else 0)

df['estrus_score'] = (
    df['cbt_above_threshold'] + df['high_activity_75'] + df['high_walking'] +
    df['high_standing'] + df['low_lying'] + df['low_feeding'] + df['milk_drop_10']
)

print(f'\n✅ Dataset final : {df.shape}')
print('\n📊 Moyenne par label :')
feat_cols = ['cbt_mean','cbt_max','walking_h','standing_h','lying_h','feeding_h','activity','milk_kg','estrus_score']
print(df.groupby('label')[feat_cols].mean().round(3))


## 5. Modèle XGBoost

### 5.1 Features & Split par vache

In [ ]:
FEATURES = [c for c in [
    'walking_h', 'activity', 'lying_h', 'standing_h', 'feeding_h',
    'cbt_mean', 'cbt_max', 'cbt_std', 'cbt_min',
    'milk_kg', 'estrus_score', 'is_night', 'hour'
] if c in df.columns]

print(f'✅ Features ({len(FEATURES)}) : {FEATURES}')
df = df.dropna(subset=FEATURES + ['label'])

label_rich = df.groupby('cow_id')['label'].sum().sort_values(ascending=False)
print(f'\n📌 Chaleur par vache :\n{label_rich}')

test_cows  = label_rich.head(3).index.tolist()
train_cows = [c for c in sorted(df['cow_id'].unique()) if c not in test_cows]

train = df[df['cow_id'].isin(train_cows)]
test  = df[df['cow_id'].isin(test_cows)]

X_train = train[FEATURES].values
y_train = train['label'].values
X_test  = test[FEATURES].values
y_test  = test['label'].values

print(f'\n🐄 Train : {len(train):,} lignes | {train_cows}')
print(f'🐄 Test  : {len(test):,} lignes  | {test_cows}')
print(f'Train label=1 : {train["label"].sum()} ({train["label"].mean()*100:.2f}%)')
print(f'Test  label=1 : {test["label"].sum()}  ({test["label"].mean()*100:.2f}%)')


### 5.2 SMOTE

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = smote.fit_resample(X_train, y_train)
print(f'⚖️  Après SMOTE : {np.bincount(y_res)}')


### 5.3 Entraînement

In [ ]:
split = int(len(X_res) * 0.85)
model = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=2.0, min_child_weight=5,
    eval_metric='logloss', early_stopping_rounds=30, random_state=42
)
print('🚀 Training...')
model.fit(
    X_res[:split], y_res[:split],
    eval_set=[(X_res[split:], y_res[split:])],
    verbose=50
)
print(f'✅ Meilleure itération : {model.best_iteration}')


## 6. Évaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print('='*55)
print(classification_report(y_test, y_pred,
      target_names=['pas chaleur','chaleur'], zero_division=0))
print(f'AUC-ROC : {roc_auc_score(y_test, y_prob):.4f}')
print('='*55)

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\n📌 Feature Importance :')
print(imp.round(4).to_string())


### 6.1 Seuils réels appris depuis les données

In [ ]:
print('📐 Seuils réels dans le dataset :')
for feat in ['walking_h','activity','lying_h','standing_h','feeding_h','cbt_mean']:
    if feat in df.columns:
        v0   = df[df['label']==0][feat].mean()
        v1   = df[df['label']==1][feat].mean()
        diff = ((v1-v0)/v0*100) if v0 > 0 else 0
        arrow = '↑' if v1 > v0 else '↓'
        print(f'   {feat:15s} normal={v0:.3f}  chaleur={v1:.3f}  {arrow}{abs(diff):.0f}%')


## 7. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Prediction Moment Optimal Insemination', fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['pas chaleur','chaleur']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matrice de Confusion')

imp.sort_values().plot(
    kind='barh', ax=axes[1],
    color=['crimson' if v > 0.15 else 'steelblue' for v in imp.sort_values()],
    edgecolor='white'
)
axes[1].set_title('Feature Importance')
axes[1].axvline(0.15, color='red', ls='--', lw=1, label='seuil 15%')
axes[1].legend(fontsize=8)

axes[2].hist(y_prob[y_test==0], bins=20, alpha=0.6, label='pas chaleur', color='steelblue')
axes[2].hist(y_prob[y_test==1], bins=20, alpha=0.6, label='chaleur',     color='crimson')
axes[2].axvline(0.5, color='black', ls='--', lw=1.5, label='seuil 0.5')
axes[2].set_title('Distribution Probabilites')
axes[2].set_xlabel('Probabilite chaleur')
axes[2].legend()

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[3].plot(fpr, tpr, color='crimson', lw=2.5, label=f'AUC = {auc:.3f}')
axes[3].fill_between(fpr, tpr, alpha=0.1, color='crimson')
axes[3].plot([0,1],[0,1], color='gray', ls='--', lw=1)
axes[3].set_xlabel('False Positive Rate')
axes[3].set_ylabel('True Positive Rate')
axes[3].set_title('Courbe ROC')
axes[3].legend(fontsize=10)

plt.tight_layout()
plt.savefig('resultats.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ resultats.png sauvegardé')


## 8. Sauvegarde du modèle

In [ ]:
joblib.dump(model,    'model.pkl')
joblib.dump(FEATURES, 'features.pkl')
print('✅ model.pkl sauvegardé')
print('✅ features.pkl sauvegardé')
print('\n🎯 Prêt pour app.py — lancer avec : streamlit run app.py')
